# Lab 18: Text to Speech

**Duration:** 3 hours. **Level:** beginner. **Work mode:** pairs or individual.

| Student name | Station no. | Date |
|---|---|---|
| | | |

## 1. Overview
In Lab 17 the Jetson learned to **hear**. In this lab it learns to **speak**. **Text to speech** (TTS) turns written text into sound. You use two voices and compare them: **espeak-ng**, a small program that builds speech from rules (it sounds like a robot), and **Piper**, a neural network that sounds much more human. Then you build an **announcer**: a program that turns the messages of your projects ("The room is full", "Two people are inside") into speech, without repeating itself all the time.

**By the end of this lab you can:**
1. Make speech from text with a rule-based voice and a neural voice, and describe the difference.
2. Change the speed and the pitch of a voice, and use another language (Malay).
3. Measure how fast a voice is with the real-time factor.
4. Clean a text so that numbers and symbols are read correctly.
5. Build an announcer with a cache and a cooldown, and play the sound on your laptop and on the Jetson speaker.

**How this notebook works.** Run a cell with **Shift + Enter**, from the top. Type your answers in the cells that say *Your answers*. A sound player appears under the cell that makes sound: press its play button. The sound also plays on the speaker of the Jetson if one is connected. Press **Ctrl + S** often.

## 2. Background

**Two ways to make a voice.**

| | espeak-ng | Piper |
|---|---|---|
| Method | Rules: the text is split into sounds (phonemes) and each sound is made by a formula | A neural network trained on hours of a real voice |
| Sound | Robot | Natural |
| Size | A few MB | About 60 MB for one voice |
| Languages | Over 100, including Malay | Many, but one file for each voice |
| Speed | Very fast | Fast on the CPU of the Jetson |

**From text to sound.** Both voices work in the same steps:
1. **Normalise the text**: `3.5 km/h` becomes "three point five kilometres per hour".
2. **Phonemes**: the words are turned into sounds. "Jetson" is spoken as *J EH T S AH N*.
3. **Synthesis**: the sounds are turned into a waveform (16 000 or 22 050 numbers per second, as in Lab 17).

Step 1 is often wrong for symbols and short forms, so a good announcer **cleans the text** before it speaks.

**Real-time factor (RTF).** As in Lab 17: `RTF = time to make the sound / length of the sound`. Below 1 means the voice is made faster than it is spoken.

**Speed and pitch.** *Speed* is the number of words per minute (normal speech is 140 to 170). *Pitch* is how high the voice is.

**A cache.** A message such as "Room is full" is spoken many times. Making it again each time wastes time. A **cache** keeps the sound file of every text, so the second time it is played at once.

**A cooldown.** An announcer that says the same warning every second is annoying and hides the other messages. After a message is spoken, the same message waits for a **cooldown** before it is spoken again.

**Change one thing at a time.** If you break the notebook, close it and run `import labview; labview.reset(18)` in a cell of a new notebook (File, New, Notebook) for a fresh copy.

---
# Part A: Prepare (25 min)

### Step 1: Check the Parts
The cell checks `espeak-ng` (the rule-based voice), `aplay` (plays on the Jetson speaker) and the Python package `piper-tts`. Setup of a new Jetson installs `espeak-ng` and `alsa-utils`. If `piper-tts` is missing, the cell installs it (about 1 minute, it needs the internet).

In [ ]:
import csv
import glob
import hashlib
import os
import re
import shutil
import subprocess
import sys
import time
import urllib.request
from datetime import datetime

import numpy as np

import labview

print("This notebook is saved in:", os.getcwd())
print("espeak-ng:", shutil.which("espeak-ng") or "MISSING (run: sudo apt install espeak-ng)")
print("aplay:    ", shutil.which("aplay") or "MISSING (run: sudo apt install alsa-utils)")
try:
    import piper
    print("piper-tts is installed.")
except ImportError:
    print("Installing piper-tts ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "piper-tts"], check=True)
    print("piper-tts is installed.")

### Step 2: Choose the Speaker
Run `labview.audio_devices()` and look at **Speakers**. The webcam microphone is not a speaker. If your Jetson speaker is not the default one, set it, for example `labview.speaker_set("plughw:1,0")`. The sound player under each cell always plays on your **laptop**, so you can hear the lab even without a Jetson speaker.

In [ ]:
labview.audio_devices()
# labview.speaker_set("plughw:1,0")      # remove the # and change the numbers if needed

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Speaker you use (or 'laptop only'):
- Did the player under the cell play on your laptop?:


---
# Part B: A Rule-Based Voice, espeak-ng (35 min)

### Step 1: Say Something
The function `synth_espeak` runs the program `espeak-ng` and saves the speech as a WAV file. `-v` is the voice (the language), `-s` is the speed in words per minute, `-p` is the pitch (0 to 99) and `-w` is the file to write. The function `speak_file` measures the time and plays the file.

In [ ]:
def synth_espeak(text, path, voice="en", speed=150, pitch=50):
    """Make speech with espeak-ng and save it as a WAV file."""
    subprocess.run(["espeak-ng", "-v", voice, "-s", str(speed), "-p", str(pitch), "-w", path, text], check=True, capture_output=True)
    return path


def speak_file(make, text, path, **options):
    """Make the sound with the function `make`, print the time and the real-time factor, and play it."""
    started = time.time()
    make(text, path, **options)
    took = time.time() - started
    sound, rate = labview.load_wav(path)
    seconds = len(sound) / rate
    print(f"{len(sound)} samples at {rate} Hz = {seconds:.1f} s of sound, made in {took:.2f} s, real-time factor {took / seconds:.2f}")
    labview.play(path)
    return took / seconds


rtf_espeak = speak_file(synth_espeak, "Hello. I am the Jetson Orin Nano. I can speak now.", "espeak_hello.wav")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Sample rate of the sound:
- Real-time factor of espeak-ng:
- How does it sound (describe it)?:


### Step 2: Speed, Pitch and Language
Change the speed and the pitch. Then try **Malay**: the voice `ms` reads Malay text.

In [ ]:
for speed in (100, 150, 220):
    print("speed", speed, "words per minute")
    synth_espeak("The room is full. Please wait outside.", f"espeak_speed{speed}.wav", speed=speed)
    labview.play(f"espeak_speed{speed}.wav")

In [ ]:
for pitch in (10, 50, 90):
    print("pitch", pitch)
    synth_espeak("Welcome to the Jetson lab.", f"espeak_pitch{pitch}.wav", pitch=pitch)
    labview.play(f"espeak_pitch{pitch}.wav")

In [ ]:
synth_espeak("Selamat datang ke makmal Jetson. Bilik ini sudah penuh.", "espeak_malay.wav", voice="ms")
labview.play("espeak_malay.wav")
print("Voices for English:", subprocess.run(["espeak-ng", "--voices=en"], capture_output=True, text=True).stdout.splitlines()[:6])

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which speed is the easiest to understand?:
- What does a low pitch and a high pitch sound like?:
- How good is the Malay speech? Can a Malay speaker understand it?:


---
# Part C: A Neural Voice, Piper (40 min)

### Step 1: Download a Voice
A Piper voice is two files: the neural network (`.onnx`, about 60 MB) and its settings (`.onnx.json`). The cell downloads the English voice *Lessac* (medium quality) once. It needs the internet.

In [ ]:
VOICE = "en_US-lessac-medium"
BASE = "https://huggingface.co/rhasspy/piper-voices/resolve/main/en/en_US/lessac/medium/"
os.makedirs("voices", exist_ok=True)
for name in (VOICE + ".onnx", VOICE + ".onnx.json"):
    path = "voices/" + name
    if os.path.exists(path):
        print(name, "is here already")
        continue
    print("Downloading", name, "...")
    urllib.request.urlretrieve(BASE + name, path)
    print(f"  {os.path.getsize(path) / 1e6:.1f} MB")

### Step 2: Speak with Piper
`synth_piper` runs the Piper program that comes with `piper-tts`. The text goes in through the input (`input=`), the speech file comes out. Compare the sound and the real-time factor with espeak-ng.

In [ ]:
def synth_piper(text, path, voice=VOICE, speed=1.0):
    """Make speech with Piper and save it as a WAV file. speed 1.0 is normal, 1.3 is slower (it is the length of each sound)."""
    command = [sys.executable, "-m", "piper", "-m", f"voices/{voice}.onnx", "-f", path]       # -m: the voice, -f: the file to write
    if speed != 1.0:
        command += ["--length-scale", str(speed)]
    result = subprocess.run(command, input=text.encode(), capture_output=True)
    if result.returncode != 0:
        raise SystemExit("Piper failed: " + result.stderr.decode(errors="replace")[-400:])
    return path


text = "Hello. I am the Jetson Orin Nano. I can speak now."
rtf_piper = speak_file(synth_piper, text, "piper_hello.wav")
print("First call includes the loading of the voice. Run the cell again to see the normal speed.")

In [ ]:
rtf_piper = speak_file(synth_piper, text, "piper_hello2.wav")
print(f"espeak-ng: real-time factor {rtf_espeak:.2f}    Piper: real-time factor {rtf_piper:.2f}")

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Real-time factor of Piper (second call) and of espeak-ng:
- Which voice sounds more natural? Give two differences you hear.:
- Is Piper fast enough for an announcer (real-time factor below 1)?:


### Step 3: Compare the Two Voices
Listen to the same three sentences with both voices and give each a score from 1 (bad) to 5 (very good) for **clearness** and **naturalness**.

In [ ]:
tests = ["The temperature is 28 degrees.", "Please count the people at the door.", "Jetson Orin Nano Super, the small computer for big AI."]
for number, sentence in enumerate(tests, 1):
    print(f"--- Sentence {number}: {sentence}")
    print("espeak-ng:")
    synth_espeak(sentence, f"cmp_espeak{number}.wav")
    labview.play(f"cmp_espeak{number}.wav")
    print("Piper:")
    synth_piper(sentence, f"cmp_piper{number}.wav")
    labview.play(f"cmp_piper{number}.wav")

| Sentence | espeak-ng clear | espeak-ng natural | Piper clear | Piper natural |
|---|---|---|---|---|
| 1 | | | | |
| 2 | | | | |
| 3 | | | | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Your scores (the table):
- Which words did a voice say wrongly (for example 'Jetson', 'Orin', numbers)?:
- When would you still choose espeak-ng and not Piper?:


---
# Part D: Build the Announcer (45 min)

### Step 1: Clean the Text
Voices read symbols badly: `28°C` may be read as "twenty eight C". The function `clean_text` replaces symbols and short forms with words before the text is spoken. This is **text normalisation**. The cell checks it.

In [ ]:
REPLACE = [
    (r"(\d+(?:\.\d+)?)\s*°\s*C\b", r"\1 degrees Celsius"),
    (r"(\d+(?:\.\d+)?)\s*%", r"\1 percent"),
    (r"(\d+(?:\.\d+)?)\s*km/h", r"\1 kilometres per hour"),
    (r"\bFPS\b", "frames per second"),
    (r"\bID\b", "I D"),
    (r"\bRM\s*(\d+(?:\.\d+)?)", r"\1 ringgit"),
    (r"&", " and "),
]


def clean_text(text):
    """Replace symbols and short forms by words, so the voice reads them correctly."""
    for pattern, replacement in REPLACE:
        text = re.sub(pattern, replacement, text)
    return " ".join(text.split())


for test in ("It is 28°C and 65% wet.", "Speed: 12.5 km/h", "The model runs at 30 FPS.", "Cat & dog, RM 12.50"):
    print(f"{test!r:34} ->", repr(clean_text(test)))

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Did all four sentences turn into good words?:
- Add one more rule of your own to `REPLACE` (write it here).:


### Step 2: The Speaker with a Cache and a Cooldown
The announcer has these parts:

| Part | What it does |
|---|---|
| `speak(text)` | Cleans the text, makes the sound (or takes it from the **cache**) and plays it |
| `announce(key, text)` | Speaks only if this `key` was not spoken in the last `COOLDOWN` seconds. Returns `True` if it spoke. |
| `log_speech(key, text, cached)` | Adds a line to `speech_log.csv` |

In [ ]:
# ---------------- Settings you can change ----------------
ENGINE = "piper"        # "piper" (natural) or "espeak" (fast and small)
COOLDOWN = 10           # seconds before the same message is spoken again
# ----------------------------------------------------------
os.makedirs("speech_cache", exist_ok=True)
last_spoken = {}


def speak(text, engine=None, play_now=True):
    """Clean the text, make the speech (or take it from the cache) and play it. Returns True when it came from the cache."""
    engine = engine or ENGINE
    text = clean_text(text)
    path = f"speech_cache/{engine}_{hashlib.md5(text.encode()).hexdigest()[:12]}.wav"
    cached = os.path.exists(path)
    if not cached:
        (synth_piper if engine == "piper" else synth_espeak)(text, path)
    if play_now:
        labview.play(path)
    return cached


def log_speech(key, text, cached):
    first = not os.path.exists("speech_log.csv")
    with open("speech_log.csv", "a", newline="") as f:
        writer = csv.writer(f)
        if first:
            writer.writerow(["time", "key", "text", "from_cache"])
        writer.writerow([datetime.now().strftime("%Y-%m-%d %H:%M:%S"), key, text, cached])


def announce(key, text, now=None):
    """Speak the message unless the same key was spoken less than COOLDOWN seconds ago."""
    now = time.time() if now is None else now
    if now - last_spoken.get(key, -1e9) < COOLDOWN:
        return False
    last_spoken[key] = now
    log_speech(key, text, speak(text))
    return True


print("The announcer is ready. Engine:", ENGINE)

### Step 3: Try the Announcer
The cell sends the same message 4 times and one different message. Only the messages that are allowed by the cooldown are spoken. The second time the same text is needed after the cooldown, it comes from the cache.

In [ ]:
t0 = time.time()
for offset, key, text in [(0, "full", "Warning. The room is full: 3 people are inside."),
                          (2, "full", "Warning. The room is full: 3 people are inside."),
                          (4, "door", "The door is open."),
                          (6, "full", "Warning. The room is full: 3 people are inside."),
                          (12, "full", "Warning. The room is full: 3 people are inside.")]:
    spoken = announce(key, text, now=t0 + offset)
    print(f"t = {offset:>2} s   {key:<5} ->", "SPOKEN" if spoken else "skipped (cooldown)")
print()
print(open("speech_log.csv").read())

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Which messages were spoken and which were skipped?:
- In the log, which lines have `from_cache` True? Why?:
- What happens to the speed of an announcement when it comes from the cache?:


### Step 4: Connect It to a Real Event
The announcer is useful when it is called by your other programs. This cell is a small example: the counter of Lab 14 (people inside) is replaced by a list of numbers. When the number reaches the limit, the announcer speaks. Change the list, the limit and the message.

In [ ]:
LIMIT = 3
inside_counts = [0, 1, 2, 3, 3, 4, 2, 1, 3]                  # what a people counter could report, second by second
t0 = time.time()
for second, inside in enumerate(inside_counts):
    if inside >= LIMIT:
        announce("room_full", f"The room is full. {inside} people are inside.", now=t0 + second * 5)
    elif inside == 0:
        announce("room_empty", "The room is empty.", now=t0 + second * 5)
print("Done.")

---
# Part E: Make It Your Own, Test and Present (35 min)

### Step 1: Make It Your Own
Add **at least one improvement** from the list, or one of your own. Change the code in the cells above, run that cell again, then run the test again.

- Use the announcer with **your own program**: call `announce(...)` from Lab 14 (people), Lab 8 (QR attendance: "Welcome, Aminah"), Lab 16 (missing item) or Lab 19 (word heard).
- Say the **time** with the greeting: `datetime.now().strftime("%I:%M %p")`.
- Add a **priority**: urgent messages (alarm) speak even during the cooldown of other messages.
- Make the announcer speak in **Malay** with espeak-ng (`voice="ms"`) and write the Malay messages.
- Download a **second Piper voice** (see https://huggingface.co/rhasspy/piper-voices) and let the user choose.
- Put the speech in a **thread** so the program does not wait while it speaks.

> **Tip.** Change one thing at a time. To keep a copy of a cell before a big change, copy the cell (select it, press **C**, then **V**) and change the copy.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Improvement you chose:
- Which lines did you change?:
- Did it work the first time? What went wrong?:


### Step 2: Test
Test each situation and write what happened.

| Test | Situation | Expected | What happened |
|---|---|---|---|
| 1 | A new message with Piper | Spoken, not from cache | |
| 2 | The same message after the cooldown | Spoken, from cache, faster | |
| 3 | The same message within the cooldown | Skipped | |
| 4 | A message with `%`, `°C` and `km/h` | Read as words | |
| 5 | A long message (30 words) | Real-time factor? | |
| 6 | A message with a name such as *Aminah* | Pronounced well? | |
| 7 | The same message with ENGINE = "espeak" | Compare with test 1 | |

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- Test 1 to 3:
- Test 4 to 5:
- Test 6 and 7:
- Which words did the voices say wrongly and how could you fix them?:


### Step 3: Report and Present
Fill in the report. Then give a **3-minute demonstration**: play both voices, show the cleaning of a text, run the announcer with cooldown and cache, and show the log.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- The two voices and their real-time factors:
- How you used the announcer:
- Power mode used (run  !sudo nvpmodel -q  in a cell, after  labview.sudo_login()  ):
- Your improvement:
- One problem you found and how you fixed it:
- One idea to make it better in future:


> **Where next.** The Jetson can now hear (Labs 17 and 19) and speak (Lab 18). In the next labs it **thinks**: a small language model runs on the Jetson and answers your questions.

## 4. Check Task
Show your instructor. The instructor initials each line when it is done correctly.

| No. | Task to demonstrate | Initials |
|---|---|---|
| 1 | The espeak-ng voice speaks and you change the speed, the pitch and the language | |
| 2 | The Piper voice is downloaded and speaks | |
| 3 | The real-time factors of both voices are measured and compared | |
| 4 | The score table of the comparison is complete | |
| 5 | `clean_text` changes symbols to words and you add one rule | |
| 6 | The announcer skips a repeated message (cooldown) and uses the cache | |
| 7 | At least one improvement works and you explain the code you changed | |
| 8 | The test table is complete | |
| 9 | The report is complete and the 3-minute presentation is given | |

## 5. Assessment Questions
Answer in one or two sentences.

**Your answers** (double-click this cell, type after each colon, then press Shift + Enter):

- 1. What is the main difference between espeak-ng and Piper?:
- 2. Name the three steps from text to sound.:
- 3. What does a real-time factor of 0.3 mean?:
- 4. Why must a text be cleaned before it is spoken? Give an example.:
- 5. What is a cache and why does the announcer use one?:
- 6. Why does the announcer use a cooldown? What would happen without it?:
- 7. Piper sounds better but is bigger. In which product would you still use espeak-ng?:
- 8. How could the Jetson say the name of a person that a voice pronounces wrongly?:
- 9. Which is a good use of text to speech? (a) A talking guide that reads the alarm of a sensor (b) Counting cars (c) Finding the edges in a picture (d) Measuring a temperature:


## 6. Troubleshooting

| Symptom | Likely cause | What to try |
|---|---|---|
| `espeak-ng: MISSING` | Setup did not install it | Run `labview.sudo_login()` and then `!sudo apt install -y espeak-ng` |
| No sound under the cell | The browser blocks sound, or the volume is low | Press the play button of the player. Check the volume of your laptop. |
| No sound from the Jetson speaker | The speaker is not the default | `labview.audio_devices()`, then `labview.speaker_set("plughw:CARD,DEVICE")`. Check the cable and the volume (`alsamixer` in a terminal). |
| `Piper failed ...` | The voice file is missing or piper-tts is not installed | Run Part A Step 1 and Part C Step 1 again |
| The voice download fails | No internet | Ask your teacher. Copy the two voice files into the `voices` folder. |
| `No module named piper` | Installation did not finish | Run Part A Step 1 again |
| The first Piper call is slow | The voice is loaded each time you call Piper | This is normal: each call starts the program. Use the cache for repeated messages. |
| A word is pronounced wrongly | The voice does not know the word | Write it as it sounds (for example 'Jetsun'), or add a rule to `REPLACE` |
| The same message is spoken again and again | `COOLDOWN` is too short, or the `key` is different each time | Raise `COOLDOWN`. Use the same key for the same kind of message. |
| The notebook looks broken | You changed or deleted too much | Close it and run `import labview; labview.reset(18)` in a cell of a new notebook (File, New, Notebook). You get a fresh copy. |

## 7. Further Reading
- Piper voices and samples: https://huggingface.co/rhasspy/piper-voices
- Piper: https://github.com/OHF-Voice/piper1-gpl
- eSpeak NG: https://github.com/espeak-ng/espeak-ng
- Text normalisation: https://en.wikipedia.org/wiki/Text_normalization